In [ ]:
!pip install mlflow

import mlflow
import mlflow.pytorch
import torch

from src.rl.env_autoscale import AutoscaleEnv
from src.rl.agent_ppo import PPOAgent


In [ ]:
def train_rl_mlflow(num_episodes=200):
    mlflow.set_experiment("orcaopta-rl-autoscale")

    env = AutoscaleEnv()
    state_dim = env._get_state().shape[0]
    action_dim = 3

    agent = PPOAgent(state_dim, action_dim)

    with mlflow.start_run():
        for episode in range(num_episodes):
            state = env.reset()
            done = False

            states, actions, rewards, log_probs = [], [], [], []

            while not done:
                action, log_prob = agent.act(state)
                next_state, reward, done, _ = env.step(action)

                states.append(state)
                actions.append(action)
                rewards.append(reward)
                log_probs.append(log_prob.detach())

                state = next_state

            policy_loss, value_loss = agent.update(
                states,
                actions,
                torch.stack(log_probs),
                rewards
            )

            ep_return = sum(rewards)
            mlflow.log_metric("episode_return", ep_return)
            mlflow.log_metric("policy_loss", policy_loss)
            mlflow.log_metric("value_loss", value_loss)

        mlflow.pytorch.log_model(agent.policy, "policy_net")
        mlflow.pytorch.log_model(agent.value, "value_net")
        mlflow.log_param("episodes", num_episodes)

    return agent



In [ ]:
agent = train_rl_mlflow(num_episodes=200)


In [ ]:
!mlflow ui
